In [1]:
import json
import os
from pathlib import Path

import pandas as pd

from analysis_utils import EXPERIMENT_NAME_MAP

In [2]:
REPO_ROOT = Path(os.path.expanduser("~/scFM_eval"))
SPLITS_DIR = REPO_ROOT / "src" / "scfm_cancer_eval" / "data_splits"
TABLE_DIR = Path("./tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def find_cv_splits(folder_path: Path) -> list[tuple[Path, str]]:
    result = []
    for root, _dirs, files in os.walk(folder_path):
        if "cv_splits.json" not in files:
            continue
        dir_name = Path(root).name
        if dir_name.startswith("__"):
            continue
        result.append((Path(root) / "cv_splits.json", dir_name))
    return sorted(result, key=lambda item: item[1])


exps = find_cv_splits(SPLITS_DIR)
exps

[(PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/brca_full/brca_cell_type/cv_splits.json'),
  'brca_cell_type'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/brca_full/brca_chemo/cv_splits.json'),
  'brca_chemo'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/brca_full/brca_outcome/cv_splits.json'),
  'brca_outcome'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/brca_full/brca_pre_post/cv_splits.json'),
  'brca_pre_post'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/brca_full/brca_subtype/cv_splits.json'),
  'brca_subtype'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/crc_mmr/cv_splits.json'),
  'crc_mmr'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/luad/luad1/cv_splits.json'),
  'luad1'),
 (PosixPath('/home/haitham/scFM_eval/src/scfm_cancer_eval/data_splits/luad/luad2/cv_splits.json'),
  'luad2'),
 (PosixPath('/ho

In [3]:
MAX_IDS_IN_TABLE = 200


def join_ids(ids) -> str:
    ids = [str(x) for x in ids]
    if len(ids) > MAX_IDS_IN_TABLE:
        return ""
    return ",".join(ids)


records = []
for fname, exp in exps:
    raw = json.loads(Path(fname).read_text())
    task = EXPERIMENT_NAME_MAP.get(exp, exp)
    for key, fold in raw.items():
        if not str(key).startswith("fold_"):
            continue
        records.append(
            {
                "experiment": exp,
                "task": task,
                "fold": key,
                "n_train": len(fold["train_ids"]),
                "n_test": len(fold["test_ids"]),
                "train_ids": join_ids(fold["train_ids"]),
                "test_ids": join_ids(fold["test_ids"]),
                "id_column": raw.get("id_column"),
                "n_splits": raw.get("n_splits"),
            }
        )

splits_df = pd.DataFrame(records)
splits_df

,experiment,task,fold,n_train,n_test,train_ids,test_ids,id_column,n_splits
0,brca_cell_type,BRCA Cell Type,fold_1,0,15747,,,cell_ids,5
1,brca_cell_type,BRCA Cell Type,fold_2,0,15747,,,cell_ids,5
2,brca_cell_type,BRCA Cell Type,fold_3,0,15747,,,cell_ids,5
3,brca_cell_type,BRCA Cell Type,fold_4,0,15747,,,cell_ids,5
4,brca_cell_type,BRCA Cell Type,fold_5,0,15747,,,cell_ids,5
5,brca_chemo,Treatment Naive vs Neoadjuvant Chemo,fold_1,31,8,"BIOKEY_1,BIOKEY_2,BIOKEY_3,BIOKEY_4,BIOKEY_5,B...","BIOKEY_10,BIOKEY_11,BIOKEY_13,BIOKEY_17,BIOKEY...",donor_id,5
6,brca_chemo,Treatment Naive vs Neoadjuvant Chemo,fold_2,31,8,"BIOKEY_1,BIOKEY_2,BIOKEY_3,BIOKEY_4,BIOKEY_7,B...","BIOKEY_5,BIOKEY_6,BIOKEY_9,BIOKEY_15,BIOKEY_26...",donor_id,5
7,brca_chemo,Treatment Naive vs Neoadjuvant Chemo,fold_3,31,8,"BIOKEY_1,BIOKEY_3,BIOKEY_5,BIOKEY_6,BIOKEY_7,B...","BIOKEY_2,BIOKEY_4,BIOKEY_12,BIOKEY_14,BIOKEY_2...",donor_id,5
8,brca_chemo,Treatment Naive vs Neoadjuvant Chemo,fold_4,31,8,"BIOKEY_1,BIOKEY_2,BIOKEY_4,BIOKEY_5,BIOKEY_6,B...","BIOKEY_3,BIOKEY_8,BIOKEY_16,BIOKEY_21,BIOKEY_2...",donor_id,5
9,brca_chemo,Treatment Naive vs Neoadjuvant Chemo,fold_5,32,7,"BIOKEY_2,BIOKEY_3,BIOKEY_4,BIOKEY_5,BIOKEY_6,B...","BIOKEY_1,BIOKEY_7,BIOKEY_19,BIOKEY_22,BIOKEY_2...",donor_id,5


In [4]:
out_path = TABLE_DIR / "Table9_data_splits.csv"
splits_df.to_csv(out_path, index=False)
print(f"Wrote {len(splits_df)} rows to {out_path}")

Wrote 43 rows to tables/Table9_data_splits.csv
